# P120 — Clasificación semisupervisada con redes convolucionales de grafo

## 1. Título y paper

**Paper:** *Semi-Supervised Classification with Graph Convolutional Networks*  
**Autoría:** Thomas N. Kipf, Max Welling  
**Año y venue:** 2017 · ICLR 2017 · arXiv:1609.02907  
**Nivel:** L2 · **Motor:** `gcn`  
**Ficha completa:** [`P120_gcn`](../../papers/foundational/P120_gcn/README.md)

**Hito:** Reduce la convolución sobre grafos a una regla de propagación de una línea, y con ella clasifica con una fracción mínima de nodos etiquetados.

- [arXiv:1609.02907](https://arxiv.org/abs/1609.02907)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Muchos datos son grafos —citas, redes sociales, moléculas— donde etiquetar es caro y solo se tiene una fracción diminuta. Los métodos previos o eran costosos en el dominio espectral, o ignoraban la estructura y solo usaban los rasgos.
2. Ejecutar una implementación mínima de la propuesta: Una aproximación de primer orden de la convolución espectral que se reduce a promediar los rasgos de cada nodo con los de sus vecinos, normalizado por el grado, y apilar dos o tres de esas capas. Nada más.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P02
- P04


## 4. Intuición

Con seis nodos etiquetados de ciento veinte, los rasgos no bastan. Pero cada nodo tiene vecinos, y promediar con ellos convierte seis etiquetas en información sobre todo el grafo.


## 5. Concepto mínimo

```text
Una capa de GCN ≈ promediar con los vecinos, normalizado por el grado:

    H' = σ( D^(-1/2) · Â · D^(-1/2) · H · W )      con Â = A + I

Apilar k capas = mirar a k saltos de distancia
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('gcn', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Cuánto acierta usando solo los rasgos?
2. ¿Cuántas capas de propagación son las mejores?
3. ¿Qué pasa con veinte?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('gcn', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('gcn', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

Sin mirar el grafo se acierta **0,447**. Con **3 capas** de propagación, **1,0**. Con **20**, cae a **0,377** — peor que no propagar y cerca del azar (0,333). La distancia entre los centros de las comunidades se hunde de **0,75** a **0,005**: un colapso de **150×**.


## 10. Comentario pedagógico

Ese colapso es el sobre-suavizado, y solo hace daño porque hay un **suelo de precisión**: el motor añade ±0,02 de ruido de medida, como tendría cualquier sistema real. Sin ese suelo, un clasificador ideal resolvería diferencias arbitrariamente pequeñas y el sobre-suavizado no se vería. Con él, apilar capas borra justamente lo que se quiere medir.


## 11. Error o anti-patrón deliberado

Anti-patrón: apilar capas de grafo como se apilan capas de una red convolucional.


In [ ]:
print('En vision, mas profundidad suele ayudar.')
print('En grafos, cada capa mezcla vecindarios mas amplios hasta igualarlo todo.')
print('Por eso las GCN que funcionan son de dos o tres capas.')

## 12. Corrección

El barrido completo, con la separación entre comunidades:


In [ ]:
r = run_paper_lab('gcn', seed=3)['result']
for fila in r['exactitud_por_numero_de_capas']:
    print(fila)
print('separacion entre comunidades:')
for fila in r['sobre_suavizado']:
    print('  ', fila)

## 13. Desafío guiado

Explica qué supuesto sobre el grafo hace que propagar ayude, y qué ocurriría en un grafo donde los vecinos tienden a ser de clase distinta.


In [ ]:
r = run_paper_lab('gcn', seed=3)['result']
show(r)

## 14. Desafío autónomo

Busca un grafo de tu trabajo —dependencias, coautoría, red interna— y mide qué fracción de las aristas une nodos de la misma clase. Decide si propagar tiene sentido ahí.


## 15. Evidencia de aprendizaje

Guarda la fracción medida y tu decisión.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P120_gcn/README.md) · evaluación formal: [`assessments/papers/P120_gcn.md`](../../assessments/papers/P120_gcn.md)


## 16. Cierre

Propagar por igual funciona si todos los vecinos son informativos. Cuando no lo son, hace falta pesarlos — es P124.


## 17. Conexión con el siguiente hito

- P124

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
